# 04 — Classification (`class_action_sought`)

Walkthrough of §3.3–§3.4 on the **binary** task. Produces:
- training + evaluation of Naive Bayes and Logistic Regression on `long_ref`
- ROC curves for both models
- confusion matrices on the test split
- SHAP top-token explanation for the LR (input to slide 8)

All artifacts land under `models/` and `results/`. The unified CLI
`python -m src.classify.train --task class_action --model {nb,lr}` does the
same thing without a notebook; this file is for the writeup / slides.

In [1]:
import sys
from pathlib import Path

# Make src importable when running from `notebooks/`
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

from src.classify.data import load_classification_data
from src.classify.classical import (
    evaluate_classifier, train_logistic_regression, train_naive_bayes,
)
from src.classify.explain import explain_lr
from src.classify.train import _plot_confusion
from src.utils import RESULTS_DIR, set_seed

set_seed(42)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/xiongfeng/miniconda3/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/xiongfeng/miniconda3/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/Users/xiongfeng/miniconda3/lib/python3.10/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/xiongfeng/miniconda3/lib/python3.10/site-packages/traitlets/config/application.py", line 992, in launch_instance
    ap

AttributeError: _ARRAY_API not found

## 1. Load data

Binary task: `class_action_sought` (Yes/No). Input text = `long_ref` (the
human-written reference summary). W8 will repeat the same flow with
`text_source='long_pred'` to compare against Yujun's BART summaries.

In [2]:
data = load_classification_data(task='class_action', text_source='long_ref')
print(f'train: {len(data.X_train)} | val: {len(data.X_val)} | test: {len(data.X_test)}')
print(f'label_names: {data.label_names}')
print(f'train positive rate: {data.y_train.mean():.3f}')

20:10:09 | src.classify.data | INFO | Loaded 1602 rows from /Users/xiongfeng/Desktop/MLDS-NLP-final/data/multilexsum_clean.parquet


20:10:09 | src.classify.data | INFO | task=class_action text_source=long_ref | train=1129 val=161 test=312 | classes=['No', 'Yes']


train: 1129 | val: 161 | test: 312
label_names: ['No', 'Yes']
train positive rate: 0.432


## 2. Train Naive Bayes (§3.3)

`ComplementNB` is the right choice for sparse TF-IDF + imbalanced classes.

In [3]:
nb_pipe = train_naive_bayes(data.X_train, data.y_train)
nb_metrics = evaluate_classifier(nb_pipe, data.X_test, data.y_test, data.label_names)
print(f'NB  | acc={nb_metrics.accuracy:.4f} f1_macro={nb_metrics.f1_macro:.4f} auc={nb_metrics.auc_roc}')

NB  | acc=0.8141 f1_macro=0.8088 auc=0.9199067558057705


## 3. Train Logistic Regression (§3.4)

L2 + `class_weight='balanced'`.

In [4]:
lr_pipe = train_logistic_regression(data.X_train, data.y_train)
lr_metrics = evaluate_classifier(lr_pipe, data.X_test, data.y_test, data.label_names)
print(f'LR  | acc={lr_metrics.accuracy:.4f} f1_macro={lr_metrics.f1_macro:.4f} auc={lr_metrics.auc_roc}')

LR  | acc=0.8942 f1_macro=0.8900 auc=0.9682002111189304


## 4. ROC curves (slide 8)

Plot both models on one axis for a clean comparison.

In [5]:
fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_estimator(nb_pipe, data.X_test, data.y_test, ax=ax, name='Naive Bayes')
RocCurveDisplay.from_estimator(lr_pipe, data.X_test, data.y_test, ax=ax, name='Logistic Regression')
ax.set_title('ROC — class_action_sought (test split)')
ax.plot([0, 1], [0, 1], linestyle='--', color='grey', linewidth=0.7)
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'roc_classaction.png', dpi=150)
plt.show()

/var/folders/_s/z9k0pk7s0_b6h8j5nn9ffdbh0000gn/T/ipykernel_19221/3592002837.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Confusion matrices

Where do the false positives / false negatives land?

In [6]:
_plot_confusion(nb_metrics.confusion, data.label_names,
                task='class_action', model='nb', split='test')
_plot_confusion(lr_metrics.confusion, data.label_names,
                task='class_action', model='lr', split='test')

20:10:14 | src.classify.train | INFO | Saved confusion matrix → /Users/xiongfeng/Desktop/MLDS-NLP-final/results/confusion_matrices/nb_classaction_test.png


20:10:14 | src.classify.train | INFO | Saved confusion matrix → /Users/xiongfeng/Desktop/MLDS-NLP-final/results/confusion_matrices/lr_classaction_test.png


PosixPath('/Users/xiongfeng/Desktop/MLDS-NLP-final/results/confusion_matrices/lr_classaction_test.png')

## 6. SHAP top tokens — LR (§3.4)

Which TF-IDF tokens push the LR toward "class action sought"? This is the
graphic that ends up on slide 8.

In [7]:
shap_path = explain_lr(lr_pipe, data.X_test, data.label_names,
                       task='class_action', top_k=15)
print(f'SHAP plot → {shap_path}')

20:10:16 | src.classify.explain | INFO | Saved SHAP plot → /Users/xiongfeng/Desktop/MLDS-NLP-final/results/lr_shap_classaction.png


SHAP plot → /Users/xiongfeng/Desktop/MLDS-NLP-final/results/lr_shap_classaction.png


## 7. Quick error-analysis hooks (feeds §4 "Error Analysis")

Which cases do the two models disagree on? Those are the most interesting
to show in the Gradio app.

In [8]:
nb_pred = nb_pipe.predict(data.X_test)
lr_pred = lr_pipe.predict(data.X_test)
disagree = pd.DataFrame({
    'case_id': data.case_ids_test,
    'y_true': data.y_test,
    'nb_pred': nb_pred,
    'lr_pred': lr_pred,
})
disagree = disagree[disagree['nb_pred'] != disagree['lr_pred']]
print(f'{len(disagree)} cases where NB and LR disagree.')
disagree.head(10)

29 cases where NB and LR disagree.


,case_id,y_true,nb_pred,lr_pred
16,CJ-NJ-0003,0,0,1
19,CJ-TN-0015,0,1,0
27,DR-DC-0009,0,1,0
97,IM-DC-0059,1,0,1
100,IM-DC-0079,1,0,1
101,IM-DC-0080,1,0,1
107,IM-MS-0003,0,1,0
150,NS-WA-0003,0,1,0
151,PA-FL-0001,1,0,1
180,PC-NV-0016,0,1,0
